# Medical QLoRA Fine-Tuning (Colab)
_Date: 2025-08-26_

I tried QLoRA fine-tuning on a medical dataset using Unsloth.

## Checking GPU

In [ ]:
!nvidia-smi

## Install required libraries

In [ ]:
!pip install -q unsloth transformers datasets accelerate bitsandbytes trl peft

## Configurations

In [ ]:
BASE_MODEL = "unsloth/llama-3-8b-bnb-4bit"
OUTPUT_DIR = "med-qlora-adapter"
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS = 1
LR = 2e-4
MAX_SEQ_LEN = 2048

## Load dataset (MedMCQA)

In [ ]:
from datasets import load_dataset

data = load_dataset("medmcqa", "all")
print(data)

## Formatting dataset for instruction style

In [ ]:
def format(example):
    q = example["question"]
    options = [example.get("opa",""), example.get("opb",""), example.get("opc",""), example.get("opd","")]
    ans = example.get("cop","A")
    text = f"Question: {q}\nA) {options[0]}\nB) {options[1]}\nC) {options[2]}\nD) {options[3]}"
    return {"instruction": text, "output": ans}

small_train = data["train"].select(range(200)).map(format)
small_val = data["validation"].select(range(50)).map(format)

print(small_train[0])

## Load base model in 4-bit and add LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0.05,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

## Tokenizing data

In [ ]:
EOS = tokenizer.eos_token

def join(ex):
    return {"text": f"<s>[INST] {ex['instruction']} [/INST] {ex['output']} {EOS}"}

train_text = small_train.map(join)
val_text = small_val.map(join)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_SEQ_LEN)

train_tok = train_text.map(tokenize, batched=True, remove_columns=train_text.column_names)
val_tok = val_text.map(tokenize, batched=True, remove_columns=val_text.column_names)

print(train_tok[0])

## Training with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    num_train_epochs=EPOCHS,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=50,
    save_steps=100,
    bf16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    max_seq_length=MAX_SEQ_LEN,
    args=args,
)

trainer.train()

## Saving adapter

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
!ls -lh $OUTPUT_DIR

## Quick test after fine-tuning

In [ ]:
def ask(q):
    text = f"<s>[INST] {q} [/INST]"
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=128)
    print(tokenizer.decode(out[0], skip_special_tokens=True))

ask("A child has barking cough and stridor at night. What is the likely diagnosis?")